In [1]:
!pip install ftfy
import pandas as pd
import os
import csv
import re
import ftfy
import unicodedata
from collections import Counter
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.trainers import BpeTrainer
from tokenizers.processors import TemplateProcessing
from tokenizers import Tokenizer
import torch
import torch.nn as nn
from torch.nn import functional as F
import numpy as np
from sklearn.model_selection import train_test_split
import time
from array import array


torch.manual_seed(42)

BATCH_SIZE     = 32
CONTEXT_SIZE   = 256
MAX_ITERS      = 50000
EVAL_INTERVAL  = 500
LR_RATE        = 3e-4
EVAL_ITERS     = 10
device = 'cuda' if torch.cuda.is_available() else 'cpu'



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.9 MB/s eta 0:00:00


# Training Code

In [3]:
tokenizer = Tokenizer.from_file(
    r'/content/sample_data/tokenizer.json'
)
tokenizer.no_padding()

In [4]:
PAD_TOKEN_ID = tokenizer.token_to_id("[PAD]")
PAD_TOKEN_ID

2

In [ ]:
# def get_tokenized_data():
#     cache_path = r'/content/sample_data/tokenized_jokes_split_nopad_3topics.npz'

#     if os.path.exists(cache_path):
#         print("Loading tokenized data from cache...")
#         data = np.load(cache_path)
#         train_data = torch.from_numpy(data['train_data'])
#         test_data  = torch.from_numpy(data['test_data'])
#     else:
#         data = pd.read_csv(r'/content/sample_data/final_cleaned_jokes_3topics.csv')
#         data = data[data.joke_cleaned.notnull()]
#         tokenizer = Tokenizer.from_file(
#             r'/content/sample_data/tokenizer.json'
#         )
#         tokenizer.no_padding()
#         sequences = [f"tell me a joke about {t} [JOKE] {j}"
#                  for t,j in zip(data.topic, data.joke_cleaned)]
#         tokenized = tokenizer.encode_batch(sequences)
#         tokenized_ids = [x.ids for x in tokenized]
#         tokenized_ids = [x for xs in tokenized_ids for x in xs]
#         tokenized_ids = torch.tensor(tokenized_ids)

#         data_len = len(tokenized_ids)
#         train_data = tokenized_ids[:int(data_len*.9)]
#         test_data  = tokenized_ids[int(data_len*.9):]
#         np.savez_compressed(
#             cache_path,
#             train_data=train_data.numpy(),
#             test_data=test_data.numpy()
#         )
#     return train_data, test_data

# train_data, test_data = get_tokenized_data()

In [15]:
def get_tokenized_data():
    cache_path = r"/content/sample_data/tokenized_jokes_split_nopad_3topics.npz"

    # if os.path.exists(cache_path):
    #     print("Loading tokenized data from cache...")
    #     data = np.load(cache_path)
    #     train_data = torch.from_numpy(data["train_data"])
    #     test_data  = torch.from_numpy(data["test_data"])
    #     return train_data, test_data

    # ---------- No cache: build it ----------
    print("Loading CSV...")
    df = pd.read_csv(r"/content/sample_data/final_cleaned_jokes_3topics.csv")
    df = df[df.joke_cleaned.notnull()].reset_index(drop=True)

    print(f"#jokes: {len(df):,}")

    print("Loading tokenizer...")
    tokenizer = Tokenizer.from_file(r"/content/sample_data/tokenizer.json")
    tokenizer.no_padding()

    # We'll store all token IDs in a compact C-style array (uint32)
    all_ids = array("I")

    batch_size = 1024  # you can tweak this if needed
    n = len(df)

    print("Tokenizing in batches...")
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch = df.iloc[start:end]

        sequences = [
            f"tell me a joke about {t} [JOKE] {j}"
            for t, j in zip(batch.topic, batch.joke_cleaned)
        ]

        encodings = tokenizer.encode_batch(sequences)

        for enc in encodings:
            all_ids.extend(enc.ids)

        if start == 0 or (start // batch_size) % 10 == 0:
            print(f"  processed {end}/{n} jokes")

    # Convert the compact array to a torch tensor
    tokenized_ids = torch.tensor(all_ids, dtype=torch.long)
    print("Total tokens:", tokenized_ids.numel())

    data_len = tokenized_ids.size(0)
    split_idx = int(data_len * 0.9)

    train_data = tokenized_ids[:split_idx]
    test_data  = tokenized_ids[split_idx:]

    # Save as compressed numpy for next time
    print("Saving cache to disk...")
    np.savez_compressed(
        cache_path,
        train_data=train_data.numpy(),
        test_data=test_data.numpy(),
    )

    return train_data, test_data


train_data, test_data = get_tokenized_data()


Loading CSV...
#jokes: 575,526
Loading tokenizer...
Tokenizing in batches...
  processed 1024/575526 jokes
  processed 11264/575526 jokes
  processed 21504/575526 jokes
  processed 31744/575526 jokes
  processed 41984/575526 jokes
  processed 52224/575526 jokes
  processed 62464/575526 jokes
  processed 72704/575526 jokes
  processed 82944/575526 jokes
  processed 93184/575526 jokes
  processed 103424/575526 jokes
  processed 113664/575526 jokes
  processed 123904/575526 jokes
  processed 134144/575526 jokes
  processed 144384/575526 jokes
  processed 154624/575526 jokes
  processed 164864/575526 jokes
  processed 175104/575526 jokes
  processed 185344/575526 jokes
  processed 195584/575526 jokes
  processed 205824/575526 jokes
  processed 216064/575526 jokes
  processed 226304/575526 jokes
  processed 236544/575526 jokes
  processed 246784/575526 jokes
  processed 257024/575526 jokes
  processed 267264/575526 jokes
  processed 277504/575526 jokes
  processed 287744/575526 jokes
  proc

In [44]:

def get_batch(context_size=CONTEXT_SIZE, batch_size=BATCH_SIZE, data='train'):
    dataset = train_data if data == 'train' else test_data
    idx = torch.randint(len(dataset) - context_size - 1, (batch_size,))

    # build batch using stacking
    x = torch.stack([dataset[i : i + context_size] for i in idx])
    y = torch.stack([dataset[i + 1 : i + 1 + context_size] for i in idx])

    x = x.to(device)
    y = y.to(device)
    return x, y
xb, yb = get_batch()
print(xb)
print(yb)

tensor([[   12,   174,    54,  ...,   202,   682,    62],
        [  403,   249,   631,  ...,   403,   249,  7358],
        [  677,  1283,    94,  ...,   191,   122,   455],
        ...,
        [   73, 27785, 16296,  ...,  2619,  1089,  1252],
        [ 1073,   111,  6079,  ...,     1,     0,   373],
        [   54,     6,  2348,  ..., 16222,    12,  1197]], device='cuda:0')
tensor([[  174,    54,     6,  ...,   682,    62,   440],
        [  249,   631,    12,  ...,   249,  7358,    12],
        [ 1283,    94,   948,  ...,   122,   455,   646],
        ...,
        [27785, 16296,   144,  ...,  1089,  1252,    74],
        [  111,  6079,   191,  ...,     0,   373,   140],
        [    6,  2348,    58,  ...,    12,  1197,    54]], device='cuda:0')


In [5]:
EOS_TOKEN_ID = tokenizer.token_to_id("[/S]")

@torch.no_grad()
def estimate_loss(context_size=CONTEXT_SIZE):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(EVAL_ITERS)
        for k in range(EVAL_ITERS):
            X, Y = get_batch(context_size=context_size,data=split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


class Head(nn.Module):
    def __init__(self, emb_dim, context_size, head_size, dropout):
        super().__init__()
        self.key   = nn.Linear(emb_dim, head_size, bias=False)
        self.query = nn.Linear(emb_dim, head_size, bias=False)
        self.value = nn.Linear(emb_dim, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(context_size, context_size)))

        self.dropout = nn.Dropout(dropout)


    def forward(self, x, attn_mask=None):
        B, T, E = x.shape
        k = self.key(x)    # (B, T, H)
        q = self.query(x)  # (B, T, H)

        weights = q @ k.transpose(-2, -1) * E**-0.5 # (B, T, H) @ (B, H, T) -> (B, T, T)
        weights = weights.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)

        if attn_mask is not None:
            pad_mask = attn_mask[:, None, :].to(weights.device)  # (B, 1, T)
            weights = weights.masked_fill(~pad_mask, float('-inf'))

        weights = F.softmax(weights, dim=-1)        # (B, T, T)
        weights = self.dropout(weights)

        v = self.value(x)  # (B, T, E)
        out = weights @ v  # (B, T, T) @ (B, T, E) -> (B, T, E)
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, emb_dim, context_size, head_size, dropout):
        super().__init__()
        self.heads = nn.ModuleList([
            Head(emb_dim, context_size, head_size, dropout) for _ in range(num_heads)
        ])
        self.proj = nn.Linear(num_heads * head_size, emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None):
        out = torch.cat([h(x, attn_mask=attn_mask) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out


class FeedForward(nn.Module):
    def __init__(self, emb_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            nn.ReLU(),
            nn.Linear(4 * emb_dim, emb_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, emb_dim, context_size, num_heads, dropout):
        super().__init__()
        head_size = emb_dim // num_heads
        self.sa   = MultiHeadAttention(num_heads, emb_dim, context_size, head_size, dropout)
        self.ffwd = FeedForward(emb_dim, dropout)
        self.ln1  = nn.LayerNorm(emb_dim)
        self.ln2  = nn.LayerNorm(emb_dim)

    def forward(self, x, attn_mask=None):
        # x + layer because of residual connections
        # deviation from paper -> pre-norm (need to test)
        x = x + self.sa(self.ln1(x), attn_mask=attn_mask)
        x = x + self.ffwd(self.ln2(x))
        return x


class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, context_size, num_att_heads, dropout):
        super().__init__()
        self.vocab_size = vocab_size
        self.emb_dim = emb_dim
        self.context_size = context_size
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(context_size, emb_dim)

        self.blocks = nn.ModuleList([
            Block(emb_dim, context_size, num_att_heads, dropout),
            Block(emb_dim, context_size, num_att_heads, dropout),
            Block(emb_dim, context_size, num_att_heads, dropout),
            Block(emb_dim, context_size, num_att_heads, dropout),
            Block(emb_dim, context_size, num_att_heads, dropout),
            Block(emb_dim, context_size, num_att_heads, dropout),
        ])
        self.ln_f = nn.LayerNorm(emb_dim)

        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        token_emb = self.token_embedding(idx) # (B, T, E)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device)) # (T, E)
        x = token_emb + pos_emb  # (B, T, E)

         # Create attention mask: True = keep, False = pad
        # attn_mask = (idx != PAD_TOKEN_ID)  # shape (B, T)
        attn_mask = None

        for block in self.blocks:
            x = block(x, attn_mask=attn_mask)
        x = self.ln_f(x)

        logits = self.lm_head(x) # (B, T, V)

        loss = None
        if targets is not None:
            # Flatten both for cross-entropy
            logits = logits.view(B * T, self.vocab_size)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets, ignore_index=PAD_TOKEN_ID)

        return logits, loss

    def generate(self, idx, max_new_tokens=256):
      for _ in range(max_new_tokens):
          idx_cond = idx[:, -self.context_size:]
          logits, _ = self(idx_cond)
          logits = logits[:, -1, :]
          probs = F.softmax(logits, dim=-1)
          idx_next = torch.multinomial(probs, num_samples=1)  # (1, 1)
          idx = torch.cat((idx, idx_next), dim=1)

          # stop if we just generated EOS
          if idx_next.item() == EOS_TOKEN_ID:
              break

      return idx


In [9]:
model = TransformerDecoder(vocab_size=30000, emb_dim=384, context_size=CONTEXT_SIZE, num_att_heads=6, dropout=0.2)
model = model.to(device)

# logits, loss = m(xb,  yb)

In [19]:
pytorch_total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
pytorch_total_params

33808944

In [ ]:
# model

In [20]:
model(xb,  yb)

(tensor([[-0.4228, -0.0093, -0.6368,  ...,  0.3113,  0.7578,  0.0413],
         [ 0.0775, -0.8908,  0.1883,  ..., -0.8052,  0.1794, -0.7253],
         [-0.5877,  0.5093,  0.3996,  ...,  0.1510,  0.5156, -1.4571],
         ...,
         [-0.0577, -0.5218,  0.1424,  ...,  0.6933,  0.3935, -1.1256],
         [-0.4403, -0.8270,  0.7678,  ..., -0.2377,  0.4133,  0.4583],
         [ 0.2428, -0.7800,  0.1435,  ...,  0.5359,  0.3350, -0.6171]],
        device='cuda:0', grad_fn=<ViewBackward0>),
 tensor(10.5226, device='cuda:0', grad_fn=<NllLossBackward0>))

In [10]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR_RATE)

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
checkpoint_dir = "/content/drive/MyDrive/decoder_checkpoints_3topics"
os.makedirs(checkpoint_dir, exist_ok=True)


In [35]:
# def get_latest_checkpoint(checkpoint_dir):
#     best_it = -1
#     best_path = None
#     for fname in os.listdir(checkpoint_dir):
#         m = re.match(r"checkpoint_(\d+)\.pt", fname)
#         if m:
#             it = int(m.group(1))
#             if it > best_it:
#                 best_it = it
#                 best_path = os.path.join(checkpoint_dir, fname)
#     return best_it, best_path

# last_it, ckpt_path = get_latest_checkpoint(checkpoint_dir)
# print("Latest checkpoint:", last_it, ckpt_path)

# checkpoint = torch.load(ckpt_path, map_location=device)
# model.load_state_dict(checkpoint["model_state_dict"])
# optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
# start_iter = checkpoint["iteration"] + 1

ckpt_path = os.path.join(checkpoint_dir, "checkpoint_29000.pt")

checkpoint = torch.load(ckpt_path, map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

start_iter = checkpoint["iteration"] + 1
print("Resuming from iteration:", start_iter)



Resuming from iteration: 29001


In [36]:
# checkpoint_dir = r'..\data\model_checkpoints_50k_testing_all_jokes'
# os.makedirs(checkpoint_dir, exist_ok=True)
a = time.time()

for it in range(start_iter, MAX_ITERS):

    xb, yb = get_batch(context_size=CONTEXT_SIZE,batch_size=BATCH_SIZE, data='train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if it % EVAL_INTERVAL == 0:
        losses = estimate_loss(CONTEXT_SIZE)
        print(f"step {it}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        ckpt_path = os.path.join(checkpoint_dir, f"checkpoint_{it}.pt")
        torch.save({
            'iteration': it,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss.item(),
        }, ckpt_path)
        print(f"[Checkpoint saved] iteration {it} -> {ckpt_path}")
        print(f"Time taken : {time.time() - a}")
        a = time.time()


step 29500: train loss 2.6476, val loss 2.8913
[Checkpoint saved] iteration 29500 -> /content/drive/MyDrive/decoder_checkpoints_3topics/checkpoint_29500.pt
Time taken : 109.89533257484436
step 30000: train loss 2.6943, val loss 2.8788
[Checkpoint saved] iteration 30000 -> /content/drive/MyDrive/decoder_checkpoints_3topics/checkpoint_30000.pt
Time taken : 109.83953428268433
step 30500: train loss 2.6206, val loss 2.8829
[Checkpoint saved] iteration 30500 -> /content/drive/MyDrive/decoder_checkpoints_3topics/checkpoint_30500.pt
Time taken : 109.70806312561035
step 31000: train loss 2.6543, val loss 2.8622
[Checkpoint saved] iteration 31000 -> /content/drive/MyDrive/decoder_checkpoints_3topics/checkpoint_31000.pt
Time taken : 109.71699452400208
step 31500: train loss 2.6227, val loss 2.8067
[Checkpoint saved] iteration 31500 -> /content/drive/MyDrive/decoder_checkpoints_3topics/checkpoint_31500.pt
Time taken : 109.71201634407043
step 32000: train loss 2.6581, val loss 2.8928
[Checkpoint s

In [50]:
idx = torch.zeros((1,1), dtype=torch.long, device=device)
res = tokenizer.decode(model.generate(idx)[0].tolist(), skip_special_tokens=False)
res

'[S] tell me a joke about berbus, physics, student [JOKE] did you hear that the rest of the physics student found out if he wasn\'t a theoretical student in physics ? they were called berbus[/S][S] tell me a joke about menstruation, english, speech [JOKE] puns format : what is menstruation called ? sexy .[/S][S] tell me a joke about johnny, principal, mitzvah [JOKE] little johnny is damn good on sunday when his classmate came home from school and asked him , " isn\'t that at home very early ? " his teacher replied , " it sounds good . " " no good . " tom replied . " bring me a drink . " the teacher answered , " no , that\'s me . " little johnny came home from school , and his parent asked , " hey , michael , do you mind if i can have a drink ? " his father said , " no , but that would be your father . " the next day jimmy was walking down the street with his mother and his old boy son . johnny ran up to his mother and asked , " mom , would i have a coke ? " his mom said , " no , that\'

## Testing

In [11]:
checkpoint_path = r"/content/drive/MyDrive/decoder_checkpoints_3topics"


checkpoint = torch.load('/content/drive/MyDrive/decoder_checkpoints_3topics/checkpoint_49500.pt', map_location='cuda')  # or 'cuda' if available

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
start_iteration = checkpoint['iteration']
loss_value = checkpoint['loss']

print(f"✅ Loaded checkpoint from iteration {start_iteration}, last loss = {loss_value:.4f}")

model.eval()

✅ Loaded checkpoint from iteration 49500, last loss = 2.6014


TransformerDecoder(
  (token_embedding): Embedding(30000, 384)
  (position_embedding): Embedding(256, 384)
  (blocks): ModuleList(
    (0-5): 6 x Block(
      (sa): MultiHeadAttention(
        (heads): ModuleList(
          (0-5): 6 x Head(
            (key): Linear(in_features=384, out_features=64, bias=False)
            (query): Linear(in_features=384, out_features=64, bias=False)
            (value): Linear(in_features=384, out_features=64, bias=False)
            (dropout): Dropout(p=0.2, inplace=False)
          )
        )
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (ffwd): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=384, out_features=1536, bias=True)
          (1): ReLU()
          (2): Linear(in_features=1536, out_features=384, bias=True)
          (3): Dropout(p=0.2, inplace=False)
        )
      )
      (ln1): LayerNorm((384,), eps=1e-05, elementwise_affine=

In [20]:
x = "tell me a joke about man, woman, friend [JOKE]"
tok_x = tokenizer.encode(x)
tok_ids = tok_x.ids[:-1]  # keep [S] ... [JOKE], drop the tokenizer's [/S]

idx = torch.tensor([tok_ids], dtype=torch.long, device=device)
out_ids = model.generate(idx, max_new_tokens=40)[0].tolist()

decoded = tokenizer.decode(out_ids, skip_special_tokens=False)

# keep everything up to the first [/S]
decoded_single = decoded.split("[/S]")[0] + "[/S]"
print(decoded_single)


[S] tell me a joke about man, woman, friend [JOKE] an old man and a 9 year old girl had been married for sitting around for about 30 years . one night the old man started to feel lonely and unable to get up , my friend had gotten back[/S]


# Tokenizer Test (Don't Run When Training)

In [ ]:
data = pd.read_csv(r'..\data\processed\final_cleaned_jokes.csv')
data = data[data.joke_cleaned.notnull()]

In [ ]:
tokenizer = Tokenizer.from_file(
    r'..\data\processed\tokenizer.json'
)
tokenizer.no_padding()

In [ ]:
data

,stable_id,joke,topic,joke_cleaned,word_count
0,0000066daaacde54c609a69be4f00c3336480137,Whoever said imitation is the sincerest form o...,"imitation, flattery, minute",whoever said imitation is the sincerest form o...,23
1,00000a79878b3729adc0e47a34dbf5d5484214c0,"You're so fat, they oughta call your dick gary...","oldman, dick, gary","you're so fat , they oughta call your dick gar...",20
2,000054bc76448f9b302e6266a72324ecb81d4083,I couldn't sleep last night. I was wondering w...,"night, sun",i couldn't sleep last night . i was wondering ...,18
3,0000b76ab05f5d5cf9952d109c5ce7b143ae016f,What's 11 & 2? The Cowboys,cowboys,what's 11 2 ? the cowboys,6
4,00015c7571451cc5eb99c24dd7bc46a5ce46b9a1,I just got done apologizing to my barbershop q...,"barbershop, quartet, bucket",i just got done apologizing to my barbershop q...,34
...,...,...,...,...,...
726782,ffff42e7a4d6a0ba5d6d548400c2786eca9609c4,There were three dinosaurs who found a magic l...,"dinosaur, shower, genie",there were three dinosaurs who found a magic l...,70
726783,ffff4cf4bb9b9e63636efbb0692d20ea00257b64,What was the hardest part about Michael Jackso...,"michael, jackson, autograph",what was the hardest part about michael jackso...,18
726784,ffff621a689a09dd38aef56fc93b58fba9efd948,My dick is called life. Life is hard,"life, dick",my dick is called life . life is hard,9
726785,ffff6e0d16c8de3d9c1c9a1100747ab55f8d6e5f,What was Jeffrey Epstein humming before dying?...,"republic, jeffrey, epstein",what was jeffrey epstein humming before dying ...,17


In [ ]:
encoded = tokenizer.encode(f"tell me a joke about {data.topic[0]}",data.joke_cleaned[0])
print(encoded.tokens)
print(encoded.ids)
print(tokenizer.decode(encoded.ids, skip_special_tokens=False))

['[S]', 'Ġtell', 'Ġme', 'Ġa', 'Ġjoke', 'Ġabout', 'Ġimitation', ',', 'Ġflattery', ',', 'Ġminute', '[JOKE]', 'Ġwhoever', 'Ġsaid', 'Ġimitation', 'Ġis', 'Ġthe', 'Ġsincerest', 'Ġform', 'Ġof', 'Ġflattery', 'Ġhasn', "'t", 'Ġhad', 'Ġa', 'Ġ7', 'yo', 'Ġmim', 'icking', 'Ġtheir', 'Ġevery', 'Ġword', 'Ġfor', 'Ġthe', 'Ġlast', 'Ġ10', 'Ġminutes', 'Ġ.', '[/S]']
[0, 373, 140, 56, 403, 249, 22485, 12, 18861, 12, 2107, 6, 4522, 234, 22485, 127, 61, 23633, 2996, 110, 18861, 3060, 148, 303, 56, 967, 4546, 16712, 2158, 387, 365, 793, 157, 61, 522, 782, 1021, 62, 1]
[S] tell me a joke about imitation, flattery, minute[JOKE] whoever said imitation is the sincerest form of flattery hasn't had a 7yo mimicking their every word for the last 10 minutes .[/S]


In [ ]:
sequences = [f"tell me a joke about {t} [JOKE] {j}"
             for t,j in zip(data.topic, data.joke_cleaned)]

In [ ]:
tokenized = tokenizer.encode_batch(sequences)

In [ ]:
tokenized_ids = [x.ids for x in tokenized]
tokenized_ids = [x for xs in tokenized_ids for x in xs]

In [ ]:
tokenized_ids = torch.tensor(tokenized_ids)

In [ ]:
tokenized_ids.shape

torch.Size([34495187])

In [ ]:
data_len = len(tokenized_ids)
train_data = tokenized_ids[:int(data_len*.9)]
test_data  = tokenized_ids[int(data_len*.9):]

In [ ]:
PAD_TOKEN_ID = tokenizer.token_to_id("[PAD]")
PAD_TOKEN_ID

2

In [ ]:
train_data

tensor([  0, 373, 140,  ...,   6, 248,  58])